# Crypto evaluation: which cipher and which signature scheme?

This notebook measures every algorithm registered in `confidential_crypto` and combines
the numbers with a sourced security scorecard into one ranking per category.
The same runners back the CLI (`uv run bench run` / `uv run bench export`), so the
notebook is for exploration; the committed decision record is `docs/crypto-evaluation.md`.

**What is measured**

| Category | Quantitative | Qualitative (scorecard, 0–3, with sources) |
|---|---|---|
| Ciphers | encrypt/decrypt throughput, peak memory, ciphertext overhead, key/nonce/tag sizes, entropy & chi-square of the ciphertext, tamper detection | AEAD, nonce-misuse resistance, random-nonce headroom, hardware independence, standardisation, library maturity, post-quantum margin |
| Signatures | keygen / sign / verify time, public key and signature size, determinism | standardisation, library maturity, misuse resistance, side-channel resistance, post-quantum, size |

> `aes-256-cbc-hmac-sha256` is the deliberate negative example (hand-rolled composition);
> the registry refuses it outside evaluation.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from bench import ciphers, export, ranking, scorecard, signers
from bench.metrics import MIB
from confidential_crypto import registry

pd.set_option("display.precision", 2)
pd.set_option("display.width", 160)

print("Ciphers :", [c.name for c in registry.available_ciphers(include_unsafe=True)])
print("Signers :", [s.name for s in registry.available_signers(include_unsafe=True)])
print("Defaults:", registry.DEFAULT_CIPHER, "/", registry.DEFAULT_SIGNER)

Ciphers : ['aes-256-gcm', 'chacha20-poly1305', 'aes-256-gcm-siv', 'xchacha20-poly1305', 'aes-256-cbc-hmac-sha256']
Signers : ['ed25519', 'ecdsa-p256', 'rsa-pss-3072', 'rsa-pss-4096', 'ml-dsa-65']
Defaults: aes-256-gcm / ed25519


## Parameters

Sizes in MiB. `ARTIFACT` may point at a real packaged model (for example the tar produced
by the producer) to add one row per cipher at the real size. Keep `REPEATS` ≥ 5 for the
committed run; lower it while exploring.

In [4]:
SIZES_MIB = [16, 256, 1024, 8192]
REPEATS = 5
MESSAGE_MIB = 16          # size of the message signed in the signature benchmark
ARTIFACT = None           # e.g. Path("../artifacts/model.tar")
MEASURE_MEMORY = True     # spawns one subprocess per (cipher, size)

## 1. Ciphers — measurements

In [ ]:
cipher_results = ciphers.run(
    [int(s * MIB) for s in SIZES_MIB],
    repeats=REPEATS,
    artifact_path=ARTIFACT,
    measure_memory=MEASURE_MEMORY,
)
view = cipher_results.assign(size_mib=cipher_results["size_bytes"] / MIB).drop(columns="size_bytes")
view[["cipher", "size_mib", "encrypt_mib_s", "decrypt_mib_s", "peak_rss_mib",
      "overhead_bytes", "nonce_bytes", "tag_bytes", "tamper_detected"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, column, title in zip(axes, ["encrypt_mib_s", "decrypt_mib_s"], ["Encrypt", "Decrypt"]):
    pivot = view.pivot(index="cipher", columns="size_mib", values=column)
    pivot.plot.bar(ax=ax, rot=20)
    ax.set_title(f"{title} throughput (MiB/s, higher is better)")
    ax.set_xlabel("")
    ax.legend(title="size (MiB)")
plt.tight_layout()

### Ciphertext statistics — a sanity check, not a criterion

Every correct cipher yields bytes indistinguishable from uniform: entropy ≈ 8.0 bits/byte and
a chi-square statistic near 255 (its degrees of freedom; anything roughly in 200–320 is
normal for these sample sizes). These numbers **cannot rank** ciphers against each other;
they only catch gross implementation errors (an ECB-like pattern, a constant nonce, plaintext
leaking through). That is why they carry no weight in the ranking below.

In [ ]:
view[["cipher", "size_mib", "entropy_bits_per_byte", "chi_square"]].pivot(
    index="cipher", columns="size_mib"
)

### Peak memory

The probe runs each (cipher, size) in a fresh interpreter and reports the RSS growth over
the interpreter baseline: plaintext + ciphertext + decrypted copy. A one-shot in-memory
design therefore needs roughly 3–4× the artifact size; this is the limit that a chunked
format (documented as future work) would remove.

In [ ]:
view.pivot(index="cipher", columns="size_mib", values="peak_rss_mib")

## 2. Ciphers — security scorecard

In [ ]:
cipher_card = scorecard.cipher_scorecard()
cipher_criteria, signer_criteria = scorecard.criteria()
display(Markdown("\n".join(f"- **{k}**: {v}" for k, v in cipher_criteria.items())))
cipher_card.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=3, subset=list(cipher_criteria))

In [ ]:
cipher_notes, signer_notes = scorecard.notes()
display(Markdown("\n\n".join(
    f"**{name}** — {entry['notes']}\n\n" + "\n".join(f"  - {s}" for s in entry["sources"])
    for name, entry in cipher_notes.items()
)))

## 3. Ciphers — weighted ranking

Weights are explicit. Security dominates on purpose: the artifact is encrypted once and
decrypted once per deployment, so a 2× throughput difference costs seconds while a
security property costs the whole design. Change the weights and re-run the cell to see
how robust the winner is.

In [ ]:
CIPHER_WEIGHTS = dict(ranking.CIPHER_WEIGHTS)   # e.g. {"encrypt_mib_s": 1, "decrypt_mib_s": 1, "peak_rss_mib": 0.5, "security_score": 3}
cipher_rank = ranking.rank(cipher_results, cipher_card, CIPHER_WEIGHTS, key="cipher")
cipher_rank

In [ ]:
ax = cipher_rank.drop(columns=["total", "production_safe"]).plot.barh(stacked=True, figsize=(10, 4))
ax.invert_yaxis()
ax.set_title("Cipher ranking — weighted, normalised contributions")
ax.set_xlabel("weighted score")
plt.tight_layout()

## 4. Signature schemes — measurements

In [ ]:
signer_results = signers.run(int(MESSAGE_MIB * MIB), repeats=REPEATS)
signer_results[["scheme", "keygen_ms", "sign_ms", "verify_ms",
                "public_key_pem_bytes", "signature_bytes", "deterministic"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
timing = signer_results.set_index("scheme")[["keygen_ms", "sign_ms", "verify_ms"]]
timing.plot.bar(ax=axes[0], rot=20, logy=True)
axes[0].set_title(f"Timing (ms, log scale, {MESSAGE_MIB} MiB message)")
axes[0].set_xlabel("")
sizes = signer_results.set_index("scheme")[["public_key_pem_bytes", "signature_bytes"]]
sizes.plot.bar(ax=axes[1], rot=20, logy=True)
axes[1].set_title("Sizes (bytes, log scale)")
axes[1].set_xlabel("")
plt.tight_layout()

## 5. Signature schemes — scorecard and ranking

In [ ]:
signer_card = scorecard.signer_scorecard()
display(Markdown("\n".join(f"- **{k}**: {v}" for k, v in signer_criteria.items())))
signer_card.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=3, subset=list(signer_criteria))

In [ ]:
display(Markdown("\n\n".join(
    f"**{name}** — {entry['notes']}\n\n" + "\n".join(f"  - {s}" for s in entry["sources"])
    for name, entry in signer_notes.items()
)))

In [ ]:
SIGNER_WEIGHTS = dict(ranking.SIGNER_WEIGHTS)
signer_rank = ranking.rank(signer_results, signer_card, SIGNER_WEIGHTS, key="scheme")
signer_rank

In [ ]:
ax = signer_rank.drop(columns=["total", "production_safe"]).plot.barh(stacked=True, figsize=(10, 4))
ax.invert_yaxis()
ax.set_title("Signature scheme ranking — weighted, normalised contributions")
ax.set_xlabel("weighted score")
plt.tight_layout()

## 6. Decision

The registry defaults must be defensible against this table. If the top-ranked
production-safe candidate differs from `DEFAULT_CIPHER` / `DEFAULT_SIGNER`, either
change the default or write the justification into the ADR (the export flags it as a
**Deviation** automatically).

In [ ]:
best_cipher = cipher_rank[cipher_rank["production_safe"]].index[0]
best_signer = signer_rank[signer_rank["production_safe"]].index[0]
print(f"top cipher : {best_cipher:24s} default: {registry.DEFAULT_CIPHER}")
print(f"top signer : {best_signer:24s} default: {registry.DEFAULT_SIGNER}")

## 7. Export the decision record

Writes `docs/crypto-evaluation.md` from the DataFrames above (same renderer as
`uv run bench export`). Commit the result together with any change to the registry defaults.

In [ ]:
from pathlib import Path

target = Path("../../docs/crypto-evaluation.md")
export.write(cipher_results, signer_results, target)
print("wrote", target.resolve())